# 24.5 设计欺诈检测系统 / Design a Fraud Detection System (Stripe / Visa / PayPal)

**中文**:欺诈检测(信用卡盗刷、支付欺诈、账户盗用)是金融科技的核心 ML 系统,也是系统设计面试的常客。它和前面几题(推荐/搜索/广告)有一个根本不同:那些系统是"排序为王",而欺诈检测是一道**极端不平衡的二分类 + 成本敏感决策**问题。它有三个鲜明的挑战:①**极端类别不平衡**——欺诈交易常常只占 0.1%~1%,准确率(accuracy)是**完全无用甚至有害**的指标("全部预测为正常"就有 99% 准确率却抓到零个欺诈);②**成本不对称**——漏掉一笔欺诈(false negative)和误拦一个好客户(false positive)的代价天差地别,所以**决策阈值必须由业务成本决定,而不是默认的 0.5**;③**对抗性**——欺诈者会不断变换手法适应你的模型(concept drift 的极端形式)。本节从零演示"为什么准确率是垃圾指标"和"如何用业务成本选阈值",再讲清欺诈系统设计的完整框架。
**English**: Fraud detection (card fraud, payment fraud, account takeover) is a core fintech ML system and a frequent system-design interview topic. It differs fundamentally from prior questions (recommendation/search/ads): those are "ranking is king," while fraud detection is an **extremely imbalanced binary classification + cost-sensitive decision** problem. It has three distinctive challenges: ① **extreme class imbalance** — fraudulent transactions are often only 0.1%–1%, making accuracy a **useless or even harmful** metric ("predict everything legit" gives 99% accuracy while catching zero fraud); ② **asymmetric cost** — the cost of missing a fraud (false negative) vs blocking a good customer (false positive) differs wildly, so the **decision threshold must be set by business cost, not the default 0.5**; ③ **adversarial** — fraudsters constantly change tactics to adapt to your model (an extreme form of concept drift). This section demonstrates "why accuracy is a garbage metric" and "how to choose a threshold by business cost" from scratch, then clarifies the complete fraud-system-design framework.

---

**中文**:**欺诈检测的三个决定性特点**:
**English**: **The three decisive traits of fraud detection**:
- **中文**:**① 极端不平衡 → 准确率是陷阱**:欺诈率 0.5% 时,一个"永远预测正常"的白痴模型就有 99.5% 准确率,却一个欺诈都抓不到。**必须用 precision / recall / PR-AUC / 成本**,绝不用 accuracy。而且要用能处理不平衡的技术:类别权重、重采样(但小心泄漏)、异常检测。
  **① Extreme imbalance → accuracy is a trap**: at 0.5% fraud, an idiot model that "always predicts legit" has 99.5% accuracy while catching zero fraud. **Must use precision / recall / PR-AUC / cost**, never accuracy. Also use imbalance-handling techniques: class weights, resampling (carefully, avoid leakage), anomaly detection.
- **中文**:**② 成本敏感 → 阈值由业务决定**:漏掉一笔 $500 的欺诈(FN)和误拦一个好客户(FP,损失一次交易 + 客户体验)代价不同——通常 **FN 远比 FP 贵**。所以决策阈值不是 0.5,而是**最小化期望业务成本**的那个点:$\text{cost}=C_{FN}\times FN+C_{FP}\times FP$。FN 越贵,阈值越低(宁可多误拦,也别漏)。
  **② Cost-sensitive → threshold set by business**: missing a $500 fraud (FN) vs blocking a good customer (FP, a lost transaction + customer friction) cost differently — usually **FN is far pricier than FP**. So the decision threshold isn't 0.5 but the point **minimizing expected business cost**: $\text{cost}=C_{FN}\times FN+C_{FP}\times FP$. The pricier FN, the lower the threshold (better over-block than miss).
- **中文**:**③ 对抗性 + 实时性**:欺诈者主动适应你的模型(你堵一个漏洞,他换个手法),所以模型会**快速失效(concept drift)**,必须持续监控和频繁重训。而且欺诈检测常要**实时**(交易发生的几十毫秒内决定放行/拦截/验证),对延迟要求极高。
  **③ Adversarial + real-time**: fraudsters actively adapt to your model (you close one hole, they switch tactics), so the model **decays quickly (concept drift)**, requiring continuous monitoring and frequent retraining. And fraud detection is often **real-time** (decide allow/block/verify within tens of milliseconds of the transaction), demanding very low latency.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 欺诈/风控系统设计, 高频）**
> **中文**:**欺诈检测=极端不平衡二分类 + 成本敏感决策 + 对抗性 + 实时**。**① 不平衡**:欺诈 <1%, **准确率无用**(全预测正常=99%准确率抓0欺诈)→用 **precision/recall/PR-AUC/成本**; 处理:类别权重、重采样(SMOTE 小心, 接 20.10)、异常检测。**② 成本敏感阈值**:阈值≠0.5, 选**最小化 C_FN×FN+C_FP×FP** 的点(FN 通常远贵→阈值调低多抓)。**③ 对抗+漂移**:欺诈者适应→模型快速失效→持续监控+频繁重训(接 22.10)。**④ 实时**:毫秒级决策(放行/拦截/加验证如二次验证)→特征平台低延迟(22.9)、流式特征。**特征**:交易金额/时间/地点、速度特征(1小时内几笔)、设备/IP、历史行为、图特征(账户关联/团伙)。**模型**:GBDT(强基线)+ 图神经网络(团伙欺诈)+ 规则引擎(可解释、快速响应新欺诈)+ 无监督异常检测(新型欺诈没标签)。**关键难题**:标签延迟(chargeback 几周后才知)、新型欺诈无标签(半监督/异常)、可解释性(合规要能解释为何拦截)、人审环路(高风险转人工)。面试金句:*"欺诈检测是极端不平衡二分类, 准确率无用要看 PR/召回/成本; 决策阈值不是0.5而是最小化业务成本(C_FN×FN+C_FP×FP, FN 通常远贵于 FP 所以阈值调低); 欺诈是对抗性的会漂移要频繁重训+监控; 实时毫秒决策(放行/拦截/加验证); 特征含速度/图/设备, 模型 GBDT+图网络+规则+异常检测, 还要处理标签延迟和可解释性。"*
> **English**: **Fraud detection = extremely imbalanced binary classification + cost-sensitive decision + adversarial + real-time**. **① Imbalance**: fraud <1%, **accuracy useless** (all-legit = 99% accuracy catching 0 fraud) → use **precision/recall/PR-AUC/cost**; handle with class weights, resampling (SMOTE carefully, per 20.10), anomaly detection. **② Cost-sensitive threshold**: threshold ≠ 0.5, choose the point **minimizing C_FN×FN + C_FP×FP** (FN usually far pricier → lower the threshold to catch more). **③ Adversarial + drift**: fraudsters adapt → model decays fast → continuous monitoring + frequent retraining (per 22.10). **④ Real-time**: millisecond decisions (allow/block/add verification like 2FA) → low-latency feature store (22.9), streaming features. **Features**: transaction amount/time/location, velocity features (how many in the last hour), device/IP, historical behavior, graph features (account linkage/rings). **Models**: GBDT (strong baseline) + graph neural networks (ring fraud) + rules engine (interpretable, fast response to new fraud) + unsupervised anomaly detection (novel fraud has no labels). **Key challenges**: label delay (chargebacks known weeks later), novel fraud unlabeled (semi-supervised/anomaly), interpretability (compliance needs to explain why blocked), human-review loop (escalate high-risk). Interview line: *"Fraud detection is extremely imbalanced binary classification; accuracy is useless, use PR/recall/cost; the decision threshold isn't 0.5 but minimizes business cost (C_FN×FN + C_FP×FP, FN usually far pricier so lower the threshold); fraud is adversarial and drifts, needing frequent retraining + monitoring; real-time millisecond decisions (allow/block/add verification); features include velocity/graph/device, models are GBDT + graph nets + rules + anomaly detection, plus handling label delay and interpretability."*


In [ ]:

# ============================================================
# 核心①:极端不平衡下准确率是垃圾指标 / core ①: accuracy is garbage under extreme imbalance
# 中文:造一个 ~0.5% 欺诈率的数据(像真实支付)。演示"全预测为正常"就有 99%+ 准确率却抓 0 个欺诈。
# English: build ~0.5% fraud data (like real payments). Show "predict all legit" gives 99%+ accuracy catching 0 fraud.
# ============================================================
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, confusion_matrix
np.random.seed(0)
N=50000
X=np.random.randn(N,12); w=np.random.randn(12)
y=(np.random.rand(N) < 1/(1+np.exp(-(X@w-8.0)))).astype(int)      # 极低欺诈率 / very low fraud rate
print(f"欺诈率 fraud rate: {y.mean():.4f}  ({y.sum()} 笔欺诈 / {N} 笔交易, 极端不平衡)")
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.4,random_state=0,stratify=y)
clf=GradientBoostingClassifier(n_estimators=150,random_state=0).fit(Xtr,ytr)
scores=clf.predict_proba(Xte)[:,1]
acc_all_legit=(yte==0).mean()
print(f"\n'全部预测为正常'的准确率 = {acc_all_legit:.4f}  ← {acc_all_legit*100:.1f}% 准确率, 但抓到 0 个欺诈!")
print("→ 准确率(accuracy)在欺诈检测里是垃圾指标: 什么都不做就有 99%+; 必须看 precision/recall/成本")


In [ ]:

# ============================================================
# 核心②:用业务成本选阈值(不是 0.5)/ core ②: choose the threshold by business cost (not 0.5)
# 中文:漏掉欺诈(FN)比误拦好客户(FP)贵得多。总成本=C_FN×FN + C_FP×FP。扫描阈值找最小成本点——它不等于 0.5。
# English: missing fraud (FN) costs far more than blocking a good customer (FP). Total cost = C_FN×FN + C_FP×FP.
#      Sweep the threshold for the minimum-cost point — it isn't 0.5.
# ============================================================
COST_FN=500   # 漏掉一笔欺诈的代价(退单/损失)/ cost of a missed fraud (chargeback/loss)
COST_FP=5     # 误拦一个好客户的代价(摩擦/流失一次交易)/ cost of blocking a good customer (friction/lost sale)
def evaluate(thr):
    pred=(scores>=thr).astype(int)
    tn,fp,fn,tp=confusion_matrix(yte,pred).ravel()
    cost=fn*COST_FN + fp*COST_FP
    return cost, precision_score(yte,pred,zero_division=0), recall_score(yte,pred,zero_division=0)
thrs=np.linspace(0.005,0.9,80)
costs=np.array([evaluate(t)[0] for t in thrs])
best_thr=thrs[costs.argmin()]
c05,p05,r05=evaluate(0.5); cb,pb,rb=evaluate(best_thr)
print(f"成本设定: 漏掉欺诈 FN=${COST_FN}, 误拦好客户 FP=${COST_FP}  → FN 比 FP 贵 {COST_FN//COST_FP} 倍")
print(f"\n{'阈值':>8}{'总成本':>12}{'精度prec':>10}{'召回rec':>9}")
print(f"{'0.50(默认)':>10}{c05:>10.0f}{p05:>10.2f}{r05:>9.2f}   ← 默认阈值漏掉太多欺诈, 成本高")
print(f"{best_thr:>8.2f}{'(最优)':>4}{cb:>8.0f}{pb:>10.2f}{rb:>9.2f}   ← 调低阈值多抓欺诈, 总成本最低")
print(f"\n最优阈值 {best_thr:.2f} ≠ 0.5!  因为 FN 太贵, 宁可牺牲精度多误拦, 也要提高召回少漏欺诈")
print("→ 决策阈值由业务成本(FN vs FP 的代价比)决定, 不是默认 0.5, 更不是最大化准确率")


In [ ]:

# ============================================================
# 可视化:成本 vs 阈值 + 欺诈系统架构 / cost vs threshold + fraud system architecture
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 成本随阈值变化 / cost vs threshold
ax[0].plot(thrs,costs,color="#4C72B0",lw=2)
ax[0].axvline(best_thr,color="#55A868",ls="--",label=f"最优阈值 {best_thr:.2f}(最低成本)")
ax[0].axvline(0.5,color="#C44E52",ls="--",label="默认阈值 0.5(成本更高)")
ax[0].scatter([best_thr],[costs.min()],color="#55A868",s=80,zorder=5)
ax[0].set_xlabel("决策阈值 threshold"); ax[0].set_ylabel("总业务成本 (C_FN×FN+C_FP×FP)")
ax[0].set_title("阈值由业务成本决定, 最优≠0.5"); ax[0].legend(fontsize=8)
# ② 欺诈系统架构 / fraud system architecture
ax[1].axis("off"); ax[1].set_title("实时欺诈检测系统",fontsize=12,weight="bold")
layers=[("交易发生(实时, 毫秒预算)","#9467BD"),("实时特征(金额/速度/设备/图, 特征平台)","#4C72B0"),
        ("模型集成(GBDT+图网络+异常检测+规则引擎)","#55A868"),("成本敏感阈值 → 放行 / 拦截 / 加验证(2FA)","#DD8452"),
        ("高风险转人工审核 → 反馈标签(但有延迟)→ 重训","#C44E52")]
for i,(l,c) in enumerate(layers):
    ax[1].add_patch(plt.Rectangle((0.05,0.8-i*0.16),0.9,0.13,fc=c,alpha=0.25,ec=c,transform=ax[1].transAxes))
    ax[1].text(0.5,0.865-i*0.16,l,ha="center",va="center",fontsize=8.5,transform=ax[1].transAxes)
    if i<4: ax[1].annotate("",xy=(0.5,0.8-i*0.16),xytext=(0.5,0.82-i*0.16),arrowprops=dict(arrowstyle="->"),transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/sd05_viz.png",dpi=80); plt.show()
print("左:总成本随阈值变化, 最优阈值(绿)远低于 0.5(红); 右:实时欺诈系统(特征→模型集成→成本阈值决策→人审反馈)")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **欺诈检测的第一课:忘掉准确率**:这是和推荐/搜索/广告最不同的地方,也是最容易在面试里露怯的点。当欺诈率是 0.5% 时,一个"什么都不做、全部放行"的模型就有 99.5% 的准确率——但它抓到了**零个欺诈**,对业务毫无价值。我们的 demo 一针见血地展示了这个陷阱。如果你在欺诈面试里说"我用准确率评估模型",基本就出局了。正确的指标是 **precision(拦截的里面多少真是欺诈)、recall(所有欺诈里抓到多少)、PR-AUC、以及最重要的——业务成本**。这个"极端不平衡下 accuracy 无用"的认知,是一切风控/欺诈/异常检测问题的起点。
2. **决策阈值是一个业务问题,不是技术问题**:模型输出的是概率(这笔交易 30% 像欺诈),但**"30% 就该拦吗?"这个决策由成本决定,不是由模型决定**。关键在于 **FN 和 FP 的代价严重不对称**:漏掉一笔欺诈可能损失几百上千美元(退单 + 声誉),而误拦一个好客户只损失一次交易 + 一点摩擦。我们的 demo 显示,当 FN 比 FP 贵 100 倍时,**最优阈值远低于 0.5**——因为"宁可多误拦几个好客户,也绝不能漏掉贵的欺诈"在成本上是划算的。这是欺诈系统设计最能体现"把 ML 和业务连接起来"的地方:**阈值不是调参调出来的,是算成本算出来的**。而且不同场景成本比不同(大额转账 vs 小额消费),所以常常是**分场景、分金额的动态阈值**。
3. **诚实的深水区:欺诈是"对抗性 + 标签延迟 + 无标签新型"的三重地狱**。①**对抗性(和所有其他 ML 系统最本质的不同)**:推荐系统的用户不会"故意骗模型",但欺诈者会——你上线一个模型堵住一种手法,他们几天内就换新花样。这意味着欺诈模型**天生会快速失效(concept drift 的极端形式)**,必须持续监控、频繁重训,还要靠规则引擎快速响应新出现的欺诈模式(规则比重训模型快)。②**标签延迟**:一笔交易是不是欺诈,可能要等几周后用户发起 chargeback 才知道——这意味着你**训练时拿不到最新的标签**,评估和重训都被延迟拖累。③**新型欺诈无标签**:全新的欺诈手法在数据里没有先例,监督模型抓不到——所以要配**无监督异常检测**(20.4)去发现"没见过的异常",以及图分析(欺诈常成团伙,账户/设备/IP 有关联)。④**可解释性**:金融监管要求你能解释"为什么拦截这笔交易",所以纯黑盒模型不够,要能给出理由(SHAP、规则)。⑤**人审环路**:高风险但不确定的交易转人工审核,人的判断再变成标签反哺模型。**结论:设计欺诈检测系统的核心不是"更强的分类器", 而是三件事——用对指标(精度/召回/成本, 忘掉准确率)、用业务成本(C_FN vs C_FP)选阈值(不是0.5)、以及应对对抗性(频繁重训+规则引擎)、标签延迟、新型欺诈(异常检测+图)、可解释性、人审环路; 这是把 ML 深度嵌入业务成本和风险决策的系统, 也是最考验'ML 与业务结合'功力的一题。**

**English**:
1. **Fraud detection's first lesson: forget accuracy**: this is the biggest difference from recommendation/search/ads and the easiest place to stumble in interviews. At 0.5% fraud, a model that "does nothing, allows everything" has 99.5% accuracy — but catches **zero fraud**, worthless to the business. Our demo cuts to this trap. If you say "I evaluate with accuracy" in a fraud interview, you're basically out. The right metrics are **precision (of what's blocked, how much is truly fraud), recall (of all fraud, how much is caught), PR-AUC, and most importantly — business cost**. This "accuracy is useless under extreme imbalance" understanding is the starting point for all risk/fraud/anomaly-detection problems.
2. **The decision threshold is a business problem, not a technical one**: the model outputs a probability (this transaction is 30% fraud-like), but **"should 30% be blocked?" is a decision set by cost, not by the model**. The key is that **FN and FP costs are severely asymmetric**: missing a fraud may lose hundreds or thousands of dollars (chargeback + reputation), while blocking a good customer loses one transaction + a little friction. Our demo shows that when FN is 100x pricier than FP, **the optimal threshold is far below 0.5** — because "better over-block a few good customers than ever miss expensive fraud" is cost-favorable. This is where fraud-system design best shows "connecting ML to business": **the threshold isn't tuned, it's computed from cost**. And different scenarios have different cost ratios (large transfers vs small purchases), so it's often **scenario- and amount-specific dynamic thresholds**.
3. **Honest deep end: fraud is the triple hell of "adversarial + label delay + unlabeled novel"**. ① **Adversarial (the most fundamental difference from all other ML systems)**: a recommender's users don't "deliberately fool the model," but fraudsters do — deploy a model plugging one tactic and they switch tactics within days. This means fraud models **inherently decay fast (an extreme form of concept drift)**, requiring continuous monitoring, frequent retraining, and a rules engine to quickly respond to newly-emerging fraud patterns (rules are faster than retraining a model). ② **Label delay**: whether a transaction is fraud may only be known weeks later when the user files a chargeback — meaning you **lack fresh labels at training time**, and both evaluation and retraining are dragged by the delay. ③ **Novel fraud is unlabeled**: a brand-new fraud tactic has no precedent in the data, so supervised models miss it — so pair with **unsupervised anomaly detection** (20.4) to find "unseen anomalies," plus graph analysis (fraud is often ring-based, with linked accounts/devices/IPs). ④ **Interpretability**: financial regulation requires you to explain "why this transaction was blocked," so a pure black box isn't enough; you must give reasons (SHAP, rules). ⑤ **Human-review loop**: high-risk but uncertain transactions escalate to human review, and human judgment becomes labels feeding back to the model. **Conclusion: designing a fraud-detection system centers not on "a stronger classifier" but on three things — use the right metrics (precision/recall/cost, forget accuracy), choose the threshold by business cost (C_FN vs C_FP, not 0.5), and handle adversariality (frequent retraining + rules engine), label delay, novel fraud (anomaly detection + graphs), interpretability, and the human-review loop; it's a system deeply embedding ML into business cost and risk decisions, and the question that best tests "combining ML with business."**

> 💼 **实战视角 / Practical angle**
> **中文**:欺诈检测落地:①**指标**:precision/recall/PR-AUC/**业务成本**(绝不用 accuracy);②**成本敏感阈值**:算 C_FN×FN+C_FP×FP 的最小点, 分场景/金额动态阈值, FN 贵就调低;③**模型集成**:GBDT(强基线)+ 图网络(团伙)+ 无监督异常检测(新型欺诈)+ **规则引擎**(可解释、快速堵新漏洞);④**特征**:速度特征(1h内几笔)、设备/IP、地理、图关联、历史行为——用低延迟**特征平台(22.9)**;⑤**实时**毫秒决策(放行/拦截/加验证 2FA), 高风险转**人审**;⑥**对抗+漂移**:持续监控(22.10)、频繁重训、A/B 新模型;⑦**可解释性**(SHAP/规则)满足合规;⑧标签延迟→用早期代理信号 + 延迟修正。**答题**:先说不平衡→别用准确率、成本敏感阈值、对抗性要频繁重训、实时+人审、异常检测抓新型、可解释性。面试金句:*"欺诈检测是极端不平衡二分类, 准确率无用要看 precision/recall/成本; 决策阈值由业务成本 C_FN×FN+C_FP×FP 决定不是0.5(FN 贵就调低多抓); 欺诈对抗性强会快速漂移要频繁重训+规则引擎快速响应; 实时毫秒决策+高风险人审; 模型 GBDT+图网络+异常检测(抓新型无标签欺诈), 还要可解释性和处理标签延迟。"*
> **English**: Fraud detection in practice: ① **metrics**: precision/recall/PR-AUC/**business cost** (never accuracy); ② **cost-sensitive threshold**: compute the minimum of C_FN×FN + C_FP×FP, dynamic per scenario/amount, lower it when FN is pricey; ③ **model ensemble**: GBDT (strong baseline) + graph nets (rings) + unsupervised anomaly detection (novel fraud) + **rules engine** (interpretable, quickly plug new holes); ④ **features**: velocity (how many in the last hour), device/IP, geography, graph linkage, historical behavior — via a low-latency **feature store (22.9)**; ⑤ **real-time** millisecond decisions (allow/block/add verification 2FA), high-risk to **human review**; ⑥ **adversarial + drift**: continuous monitoring (22.10), frequent retraining, A/B new models; ⑦ **interpretability** (SHAP/rules) for compliance; ⑧ label delay → use early proxy signals + delayed correction. **Answering**: first raise imbalance → don't use accuracy, cost-sensitive threshold, adversarial needs frequent retraining, real-time + human review, anomaly detection for novel fraud, interpretability. Interview line: *"Fraud detection is extremely imbalanced binary classification; accuracy is useless, use precision/recall/cost; the decision threshold is set by business cost C_FN×FN + C_FP×FP not 0.5 (lower it when FN is pricey to catch more); fraud is highly adversarial and drifts fast, needing frequent retraining + a rules engine for fast response; real-time millisecond decisions + high-risk human review; models are GBDT + graph nets + anomaly detection (catch novel unlabeled fraud), plus interpretability and handling label delay."*

---
### 小结 / Summary
- **中文**:欺诈检测=极端不平衡二分类+成本敏感决策+对抗性+实时; 准确率无用(全预测正常=99%准确率抓0欺诈), 用 precision/recall/成本。
- **English**: Fraud detection = extremely imbalanced binary classification + cost-sensitive decision + adversarial + real-time; accuracy useless (all-legit = 99% catching 0), use precision/recall/cost.
- **中文**:决策阈值由业务成本 C_FN×FN+C_FP×FP 决定(不是0.5); FN 通常远贵→阈值调低多抓, 分场景动态阈值。
- **English**: The decision threshold is set by business cost C_FN×FN + C_FP×FP (not 0.5); FN usually far pricier → lower the threshold to catch more, dynamic per scenario.
- **中文**:难点:对抗性快速漂移(频繁重训+规则引擎)、标签延迟、新型欺诈无标签(异常检测+图)、可解释性、实时+人审环路。
- **English**: Difficulties: adversarial fast drift (frequent retraining + rules engine), label delay, unlabeled novel fraud (anomaly detection + graphs), interpretability, real-time + human-review loop.
